In [1]:
# pip install tiktoken matplotlib

In [2]:
import torch
import tiktoken
from configs.model import ModelConfig
from configs.training import TrainingConfig
from configs.scheduler import SchedulerConfig
from configs.optimizer import OptimizerConfig
from configs.checkpoint import CheckpointConfig
from models.qwen import Qwen
from datasets.preprocess import download_the_verdict,train_val_dataloader
from evaluation.losses import cross_entropy_loss,token_accuracy
from trainer.trainer import Trainer

c:\Users\admin\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
tokenizer = tiktoken.get_encoding("gpt2")

In [4]:
Qwen_SMALL = ModelConfig(
    emb_dim=96,
    n_layers=2,
    n_heads=12,
    kv_heads=6,
    activation="gelu",
    context_length=24
)

TRAIN_CONFIG = TrainingConfig(
    epoch=5,
    batch_size=2,
    stride=24,
    context_length=24
)

OPTIMIZER_CONFIG = OptimizerConfig()
SCHEDULER_CONFIG = SchedulerConfig()
CHECKPOINT_CONFIG = CheckpointConfig()


In [5]:
raw_text = download_the_verdict()
train_dataloader,val_dataloader = train_val_dataloader(raw_text, TRAIN_CONFIG)

TrainingConfig(epoch=5, batch_size=2, stride=24, context_length=24, learning_rate=0.0003, weight_decay=0.1, grad_clip=1.0, mixed_precision=False, shuffle=False, num_workers=0, drop_last=True, gradient_accumulation_steps=1, train_data_ratio=0.9)
TrainingConfig(epoch=5, batch_size=2, stride=24, context_length=24, learning_rate=0.0003, weight_decay=0.1, grad_clip=1.0, mixed_precision=False, shuffle=False, num_workers=0, drop_last=True, gradient_accumulation_steps=1, train_data_ratio=0.9)


In [6]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cpu


In [7]:
model = Qwen(Qwen_SMALL).to(device)

In [8]:
trainer=Trainer(model,tokenizer,train_dataloader,val_dataloader,device,cross_entropy_loss,token_accuracy,CHECKPOINT_CONFIG,TRAIN_CONFIG,OPTIMIZER_CONFIG,SCHEDULER_CONFIG)

In [9]:
trainer.fit()           ### use arg resume_latest=True or resume_best=True to resume training

100%|██████████| 96/96 [00:20<00:00,  4.61it/s]


Checkpoint saved -> checkpoints\checkpoint_96.pt
Best checkpoint saved -> checkpoints\best_checkpoint.pt
Output text:
 Every effort moves you selecting
after 1 epoch global step 96 the train loss 10.202888896067938 val loss 8.553470611572266 and train acc| 0.04144965313995878 val acc| 0.03787878528237343 


 48%|████▊     | 46/96 [00:13<00:14,  3.53it/s]


KeyboardInterrupt: 